In [ ]:
import os
import joblib
import numpy as np
from dataclasses import dataclass

In [ ]:
MODEL_PATH = "maternal_rf_model.pkl"

if not os.path.exists(MODEL_PATH):
    raise RuntimeError(
        f"Model file not found at {MODEL_PATH}. Run maternal_mortality_rf.py first.")

model_bundle = joblib.load(MODEL_PATH)
best_rf = model_bundle["model"]
le = model_bundle["label_encoder"]

NameError: name 'os' is not defined

In [ ]:
RISK_MODELS = {
    "random_forest": {"model": best_rf, "label_encoder": le},
}

EXTRA_CLASSIFIERS = {
    "xgboost": "xgb_model.pkl",
    "svm": "svm_model.pkl",
    "logistic_regression": "lr_model.pkl",
}

for name, path in EXTRA_CLASSIFIERS.items():
    if os.path.exists(path):
        bundle = joblib.load(path)
        RISK_MODELS[name] = bundle

In [ ]:
MODEL2_PATHS = {
    h: f"maternal_rf_future_t{h}.pkl"
    for h in [1, 3, 5, 10]
}

FUTURE_MODELS = {}
for h, path in MODEL2_PATHS.items():
    if os.path.exists(path):
        bundle = joblib.load(path)
        FUTURE_MODELS[h] = bundle

REGION_LABEL_MAP = {
    "Sub-Saharan Africa": 0,
    "South/SE Asia": 1,
    "Latin America": 2,
    "Developed": 3,
}

TEMPORAL_MEDIANS = {
    "deaths_lag1": 1200.0,
    "deaths_lag2": 1250.0,
    "deaths_lag3": 1300.0,
    "mmr_lag1": 280.0,
    "mmr_lag2": 295.0,
    "deaths_roll3_mean": 1250.0,
    "deaths_roll5_mean": 1280.0,
    "deaths_yoy_change": -0.03,
}

In [ ]:
@dataclass
class PatientData:
    age: int
    systolic_bp: int
    diastolic_bp: int
    blood_sugar: float
    body_temp: float
    heart_rate: int
    model_name: str = "random_forest"


@dataclass
class FutureDeathsRequest:
    horizon_years: int
    region: str
    gdp_per_capita: float
    health_expenditure_pct: float
    skilled_birth_attend: float
    antenatal_care_pct: float
    female_literacy_rate: float
    physicians_per_1000: float
    hospital_beds_per_1000: float
    urban_population_pct: float
    contraceptive_use_pct: float
    total_births: int

In [ ]:
def build_patient_features(data: PatientData) -> np.ndarray:
    return np.array([[
        data.age,
        data.systolic_bp,
        data.diastolic_bp,
        data.blood_sugar,
        data.body_temp,
        data.heart_rate,
    ]], dtype=np.float64)


def clinical_flags(data: PatientData) -> list:
    flags = []
    if data.systolic_bp >= 140:
        flags.append(f"Hypertension Stage 2 (SBP: {data.systolic_bp} ≥ 140)")
    if data.diastolic_bp >= 90:
        flags.append(f"Hypertension Stage 2 (DBP: {data.diastolic_bp} ≥ 90)")
    if data.blood_sugar >= 11:
        flags.append(
            f"High Blood Glucose (BS: {data.blood_sugar} ≥ 11 mmol/L)")
    if data.body_temp >= 100:
        flags.append(f"Fever (Temp: {data.body_temp} ≥ 100 °F)")
    if data.heart_rate >= 100:
        flags.append(f"Tachycardia (HR: {data.heart_rate} ≥ 100 bpm)")
    return flags if flags else ["No immediate red flags detected."]

In [ ]:
def predict(data: PatientData):
    if data.model_name not in RISK_MODELS:
        raise ValueError(
            f"Model '{data.model_name}' is not loaded. Available: {list(RISK_MODELS.keys())}")

    bundle = RISK_MODELS[data.model_name]
    model = bundle["model"]
    encoder = bundle["label_encoder"]

    X_input = build_patient_features(data)
    pred_idx = model.predict(X_input)[0]
    pred_class = encoder.classes_[pred_idx]

    if hasattr(model, "predict_proba"):
        pred_proba = model.predict_proba(X_input)[0]
        probabilities = {
            cls: round(float(p), 4)
            for cls, p in zip(encoder.classes_, pred_proba)
        }
    else:
        probabilities = {
            "note": "This model does not support probability output"}

    recommendation = {
        "high risk": "Urgent specialist referral required.",
        "mid risk": "Enhanced monitoring recommended.",
        "low risk": "Routine monitoring.",
    }.get(pred_class, "Routine monitoring.")

    return {
        "model_used": data.model_name,
        "predicted_risk": pred_class,
        "probabilities": probabilities,
        "clinical_flags": clinical_flags(data),
        "recommendation": recommendation,
    }

In [ ]:
def predict_future(req: FutureDeathsRequest):
    if req.horizon_years not in FUTURE_MODELS:
        raise ValueError(
            f"Model for horizon t+{req.horizon_years} not loaded.")

    bundle = FUTURE_MODELS[req.horizon_years]
    model = bundle["model"]
    region_enc = REGION_LABEL_MAP.get(req.region, 0)

    micro_defaults = {
        "avg_age": 28.0, "avg_parity": 2.8,
        "avg_systolic_bp": 115.0, "avg_diastolic_bp": 75.0,
        "avg_blood_glucose": 8.8, "avg_hemoglobin": 11.2,
        "avg_bmi": 23.5, "avg_antenatal_visits": 4.2,
        "pct_skilled_attendant": 0.62, "pct_prior_complications": 0.12,
        "pct_rural": 0.52, "avg_risk_score": 0.41,
    }

    row = [
        req.gdp_per_capita, req.health_expenditure_pct, req.skilled_birth_attend,
        req.antenatal_care_pct, req.female_literacy_rate, req.physicians_per_1000,
        req.hospital_beds_per_1000, req.urban_population_pct, req.contraceptive_use_pct,
        req.total_births, region_enc,
        micro_defaults["avg_age"], micro_defaults["avg_parity"],
        micro_defaults["avg_systolic_bp"], micro_defaults["avg_diastolic_bp"],
        micro_defaults["avg_blood_glucose"], micro_defaults["avg_hemoglobin"],
        micro_defaults["avg_bmi"], micro_defaults["avg_antenatal_visits"],
        micro_defaults["pct_skilled_attendant"], micro_defaults["pct_prior_complications"],
        micro_defaults["pct_rural"], micro_defaults["avg_risk_score"],
        TEMPORAL_MEDIANS["deaths_lag1"], TEMPORAL_MEDIANS["deaths_lag2"],
        TEMPORAL_MEDIANS["deaths_lag3"], TEMPORAL_MEDIANS["mmr_lag1"],
        TEMPORAL_MEDIANS["mmr_lag2"], TEMPORAL_MEDIANS["deaths_roll3_mean"],
        TEMPORAL_MEDIANS["deaths_roll5_mean"], TEMPORAL_MEDIANS["deaths_yoy_change"],
    ]

    X = np.array(row, dtype=np.float64).reshape(1, -1)
    pred = float(model.predict(X)[0])

    tree_preds = np.array([t.predict(X)[0] for t in model.estimators_])
    ci_low = max(0, float(np.percentile(tree_preds, 5)))
    ci_high = float(np.percentile(tree_preds, 95))

    flags = []
    if req.skilled_birth_attend < 60:
        flags.append("Low skilled birth attendance (<60%)")
    if req.antenatal_care_pct < 70:
        flags.append("Low antenatal care coverage (<70%)")
    if req.female_literacy_rate < 60:
        flags.append("Low female literacy (<60%)")
    if req.gdp_per_capita < 1000:
        flags.append("Very low GDP per capita (<$1,000)")
    if req.physicians_per_1000 < 0.5:
        flags.append("Critical physician shortage (<0.5 per 1,000)")

    return {
        "predicted_deaths": round(pred),
        "confidence_interval": [round(ci_low), round(ci_high)],
        "policy_flags": flags if flags else ["No critical policy thresholds breached."],
    }